# Ch.4　時系列データの異常検知（演習）

このノートブックは **穴埋め形式** です。  
`___` の部分を手入力するか、Copilot Chat でコードを生成して埋めてください。

---

### Ch.3 との接続

```
Ch.3: y ＝ 品種ラベル  → RandomForest で分類    → 正解率で評価
Ch.4: y ＝ 次の時刻の値 → LinearRegression で予測 → 予測誤差を異常スコアに使う
```

学習（fit）→ 予測（predict）の流れは Ch.3 とまったく同じです。  
違いは「y に渡すものが品種ではなく次の時刻の値」という点だけです。

---

> **🤖 Copilot Chat**  
> ブラウザで https://copilot.microsoft.com を開いて使ってください。
> ```
> 【やりたいこと】〇〇したい
> 【使うライブラリ】numpy、sklearn 0.23.2（LinearRegression）、pandas
> 【データの形】300行 × 2列（time, value）の時系列 DataFrame
> 【環境】Python 3.8.6、Windows、JupyterLab
> 【わからない点】〇〇の書き方
> ```

> ✅ **問3まで完了すれば十分です。問4は発展問題です。**

---
## 0. ライブラリの読み込み（完成済み・実行するだけ）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
from sklearn.linear_model import LinearRegression

print('ライブラリの読み込み完了')

---
## 1. 時系列データを生成する（完成済み・実行するだけ）

numpy で時系列データを生成します。  
3か所（時刻 80 / 160 / 240）に異常スパイクを混入しています。  
後の手順でこの3点を正しく検出できるか確認します。

In [ ]:
np.random.seed(42)
n = 300
t = np.arange(n)

values = np.sin(2 * np.pi * t / 30) * 10 + np.random.normal(0, 1, n)

anomaly_idx = [80, 160, 240]
for idx in anomaly_idx:
    values[idx] += np.random.uniform(15, 25)

df = pd.DataFrame({'time': t, 'value': values})
print('データ形状:', df.shape)
print(f'混入した異常の位置: {anomaly_idx}')

---
## 2. 時系列データを可視化する（完成済み・実行するだけ）

グラフを見て、赤い点（異常スパイク）が目でわかることを確認してください。

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['time'], df['value'], color='steelblue', linewidth=1, label='観測値')
for idx in anomaly_idx:
    plt.axvline(x=idx, color='red', linestyle='--', alpha=0.5)
plt.scatter(anomaly_idx, df['value'].iloc[anomaly_idx],
            color='red', zorder=5, s=80, label='異常（混入済み）')
plt.xlabel('時間')
plt.ylabel('観測値')
plt.title('時系列データ（赤破線 = 異常を混入した時点）', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

---
## 問1 ★☆☆　ラグ特徴量を作成する

「1つ前・2つ前・3つ前の値」を説明変数として追加します。

```
時刻  t   の値 → 予測したい（目的変数）
時刻 t-1  の値 → lag1
時刻 t-2  の値 → lag2  } 説明変数
時刻 t-3  の値 → lag3
```

`.shift(N)` は値を N 行ずつ下にずらす操作です。

**期待する出力:**
```
ラグ特徴量追加後の形状: (297, 5)
```

In [ ]:
df['lag1'] = df['value'].shift(___)  # 1つ前の値
df['lag2'] = df['value'].shift(___)  # 2つ前の値
df['lag3'] = df['value'].shift(___)  # 3つ前の値
df = df.dropna().reset_index(drop=True)

print('ラグ特徴量追加後の形状:', df.shape)
df.head()

---
## 問1（続き）　訓練データとテストデータに分割する（完成済み・実行するだけ）

時系列データは **時間の順番を守って** 分割します。

In [ ]:
split_point = 200

X_cols = ['lag1', 'lag2', 'lag3']
X_train = df[X_cols][:split_point].values
X_test  = df[X_cols][split_point:].values
y_train = df['value'][:split_point].values
y_test  = df['value'][split_point:].values

print('訓練データ:', X_train.shape)
print('テストデータ:', X_test.shape)

---
## 問2 ★★☆　LinearRegression で学習・予測する

Ch.3 の RandomForest と同じ手順です。

1. `LinearRegression()` でモデルを作成する
2. `.fit()` で訓練データを学習させる
3. `.predict()` で訓練・テスト両方の予測値を計算する

In [ ]:
model = LinearRegression()
model.___(X_train, y_train)  # 学習メソッドを入れてください

y_train_pred = model.___(X_train)  # 予測メソッドを入れてください
y_test_pred  = model.___(X_test)   # 予測メソッドを入れてください
y_pred_all   = np.concatenate([y_train_pred, y_test_pred])

print('学習完了')

---
## 問2（続き）　異常スコアを計算する

**異常スコア = |観測値 − 予測値|**

- 予測が当たった → スコアが小さい（正常）
- 予測から大きく外れた → スコアが大きい（異常の可能性）

In [ ]:
y_actual = df['value'].values
anomaly_score = np.abs(___ - ___)  # 観測値と予測値の差の絶対値を計算してください

print('異常スコアの基本統計:')
print(pd.Series(anomaly_score).describe().round(2))

---
## 問3 ★★☆　しきい値を設定して異常を判定する

`np.percentile(配列, パーセンタイル)` でしきい値を計算します。  
「上位 2%（=下位 98%tile）を異常とみなす」場合:

```python
threshold = np.percentile(anomaly_score, 98)  # 上位2%のしきい値
```

**期待する出力:**
```
しきい値: 8.370
異常と判定した件数: 6 件
```

In [ ]:
contamination = 0.02  # 異常とみなす割合（2%）
threshold = np.percentile(anomaly_score, ___)  # パーセンタイルを計算してください

is_anomaly = anomaly_score ___ threshold  # しきい値を超えた点を True にしてください

print(f'しきい値: {threshold:.3f}')
print(f'異常と判定した件数: {is_anomaly.sum()} 件')
print(f'異常と判定した時点: {df["time"][is_anomaly].tolist()}')
print(f'混入した異常の時点: {anomaly_idx}')

---
## 問3（続き）　結果を可視化する（完成済み・実行するだけ）

上段: 観測値と予測値、下段: 異常スコアとしきい値を表示します。

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.plot(df['time'], y_actual, label='観測値', color='steelblue', lw=1)
ax1.plot(df['time'], y_pred_all, label='予測値', color='orange', lw=1, linestyle='--')
ax1.scatter(df['time'][is_anomaly], y_actual[is_anomaly],
            color='red', zorder=5, s=80, label='異常検知')
ax1.set_ylabel('観測値')
ax1.set_title('時系列の異常検知結果', fontsize=13)
ax1.legend()
ax1.grid(lw=0.4)

ax2.plot(df['time'], anomaly_score, color='coral', lw=1, label='異常スコア')
ax2.axhline(y=threshold, color='red', linestyle='--',
            label=f'しきい値: {threshold:.2f}')
ax2.scatter(df['time'][is_anomaly], anomaly_score[is_anomaly],
            color='red', zorder=5, s=80)
ax2.set_xlabel('時間')
ax2.set_ylabel('異常スコア（|観測値 − 予測値|）')
ax2.legend()
ax2.grid(lw=0.4)

plt.tight_layout()
plt.show()

### 3-1. 考察してみる（自分の言葉で書く）

グラフと出力を見て気づいたことを書いてください。

**ヒント:**
- 混入した3か所（時刻 80 / 160 / 240）を正しく検出できましたか？
- 誤検知（正常なのに異常と判定）はありましたか？
- `contamination` の値を変えたらどう変わりそうですか？

In [ ]:
# ここに考察をコメントとして書いてください
# 例:
# - 時刻 80 / 160 / 240 のスパイクが検出できた
# - contamination=0.02 のとき、誤検知は〇件あった
# - contamination を小さくすると...


---
## 問4 ★★★　しきい値を変えて比較する（発展）

`contamination` を 1% / 5% / 10% に変えて、検出件数がどう変わるか確認してください。

**考えてみてほしいこと:**
- 見逃し（異常なのに正常と判定）と誤検知のトレードオフはどう変化しましたか？
- この時系列データに最も適した `contamination` は何%だと思いますか？

In [ ]:
# ここにコードを書いてください
# for rate in [0.01, 0.05, 0.10]:
#     th = np.percentile(anomaly_score, ...)
#     count = ...
#     print(...)


---
## まとめ

```
① 時系列データを用意する
      ↓
② ラグ特徴量を作る（shift で過去の値を列に追加）
      ↓
③ 正常期間で LinearRegression を学習させる
      ↓
④ 全期間に対して予測値を計算する
      ↓
⑤ 異常スコア = |観測値 − 予測値| を計算する
      ↓
⑥ しきい値を超えた時点を「異常」と判定する
```

> **完成版ノートブック**は `ch4_anomaly_detection.ipynb` に格納されています。  
> わからなかった箇所は完成版と比較して確認してください。